# Neural Network for Regression with One and Two Hidden Layers

This notebook demonstrates how to build and train a neural network for a regression task using PyTorch. 

We will create two models: one with a single hidden layer and another with two hidden layers, and compare their performance on a synthetic dataset.

## 1. Importing Necessary Libraries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## 2. Generating Synthetic Data
We will create a synthetic dataset based on a sine wave with some added noise. 

This will give us a simple non-linear relationship for our models to learn.

In [ ]:
# Generates 100 evenly spaced numbers between 0 and 2π (approximately 6.28). 
# This range is chosen because it represents one complete cycle of a sine wave.
X = np.linspace(0, 2 * np.pi, 100).reshape(-1, 1) 

y = np.sin(X) + 0.1 * np.random.randn(100, 1)

X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).float()

plt.scatter(X, y)
plt.title("Synthetic Data for Regression")
plt.xlabel("X")
plt.ylabel("y")
plt.show()

## 3. Model with One Hidden Layer

### Defining the Network Architecture
We define a simple neural network with one input neuron, one hidden layer with 10 neurons, and one output neuron. 

We are using the Tanh activation function in the hidden layer.

In [ ]:
class NetOneHidden(nn.Module):
    def __init__(self):
        super(NetOneHidden, self).__init__()
        self.fc1 = nn.Linear(1, 10) # Input layer to hidden layer
        self.tanh = nn.Tanh()      # Activation function
        self.fc2 = nn.Linear(10, 1) # Hidden layer to output layer

    def forward(self, x):
        x = self.fc1(x)
        x = self.tanh(x)
        x = self.fc2(x)
        return x

### Training the Model
We will train the model using the Mean Squared Error (MSE) loss function and the Adam optimizer.
- Try different optimizers
- Play with HyperParameters

In [ ]:
model_one_hidden = NetOneHidden() # creates an instance of the one-hidden-layer network defined earlier.
criterion = nn.MSELoss() # uses mean squared error as the regression loss.
optimizer = torch.optim.Adam(model_one_hidden.parameters(), lr=0.01) # uses the Adam optimizer to update the network parameters with learning rate 0.01.

for epoch in range(1000):
    y_pred = model_one_hidden(X_tensor) # forward pass — the model takes the input tensor X_tensor (shape [100,1]) and returns predictions (shape [100,1]).
    loss = criterion(y_pred, y_tensor) # computes the MSE loss between predictions and targets y_tensor.
    optimizer.zero_grad() # clears any gradients from the previous step (important to avoid gradient accumulation).
    loss.backward() # backpropagates the loss to compute gradients for all model parameters.
    optimizer.step() # updates the model parameters using the computed gradients and Adam's update rule.
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/1000], Loss: {loss.item():.4f}')

In [ ]:
model_one_hidden

In [ ]:
# We can also see the learned parameters. These are based on the SCALED data.
# model_one_hidden.parameters() yields multiple tensors (fc1.weight, fc1.bias, fc2.weight, fc2.bias),
params = list(model_one_hidden.parameters())
print(f'Total parameter tensors: {len(params)}')
for i, p in enumerate(params):
	print(f'param[{i}] shape={tuple(p.shape)}')

# Final layer's weight and bias (fc2):
w_final = model_one_hidden.fc2.weight.detach().numpy()
b_final = model_one_hidden.fc2.bias.detach().numpy()
print(f'Final layer weight (fc2): {w_final}, bias (fc2): {b_final}')

### Visualizing the Results

**Don't forget** to detach the tensor from the computation graph and convert it to a NumPy array for plotting.

In [ ]:
predicted_one_hidden = model_one_hidden(X_tensor).detach().numpy() 
plt.scatter(X, y, label='Original data')
plt.plot(X, predicted_one_hidden, label='Fitted line (1 hidden layer)', color='r')
plt.title("Regression with One Hidden Layer")
plt.xlabel("X")
plt.ylabel("y")
plt.legend()
plt.show()

## 4. Model with Two Hidden Layers

### Defining the Network Architecture
Now, let's create a deeper network with two hidden layers. 

The first hidden layer will have 10 neurons, and the second will have 5 neurons.

In [ ]:
class NetTwoHidden(nn.Module):
    def __init__(self):
        super(NetTwoHidden, self).__init__()
        self.fc1 = nn.Linear(1, 15)      # Input to first hidden layer
        self.tanh1 = nn.Tanh()
        # self.fc2 = nn.Linear(10, 15)      # First hidden to second hidden layer
        # self.tanh2 = nn.Tanh()
        # self.fc3 = nn.Linear(15, 15)      # First hidden to second hidden layer
        # self.tanh3 = nn.Tanh()
        self.fc4 = nn.Linear(15, 1)       # Second hidden to output layer

    def forward(self, x):
        x = self.fc1(x)
        x = self.tanh1(x)
        # x = self.fc2(x)
        # x = self.tanh2(x)
        # x = self.fc3(x)
        # x = self.tanh3(x)
        x = self.fc4(x)
        return x

In [ ]:
print(f'X shape: {X_tensor.shape}, y shape: {y_tensor.shape}')
print(f'X_tensor: {X_tensor[:10]}')
print(f'y_tensor: {y_tensor[:10:]}')

### Training the Model

In [ ]:
model_two_hidden = NetTwoHidden()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_two_hidden.parameters(), lr=0.01)

for epoch in range(1000):
    y_pred = model_two_hidden(X_tensor)
    loss = criterion(y_pred, y_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/1000], Loss: {loss.item():.4f}')

### Visualizing and Comparing the Results

In [ ]:
predicted_two_hidden = model_two_hidden(X_tensor).detach().numpy()
plt.scatter(X, y, label='Original data')
plt.plot(X, predicted_one_hidden, label='Fitted line (1 hidden layer)', color='r')
plt.plot(X, predicted_two_hidden, label='Fitted line (2 hidden layers)', color='g')
plt.title("Comparison of Models")
plt.xlabel("X")
plt.ylabel("y")
plt.legend()
plt.show()

### Try experimenting by: 
- Adding more layers or neurons. 
- Changing activation functions (e.g., `ReLU`). 
- Modifying the learning rate or noise level.

## 5. Conclusion
In this notebook, we have successfully built and trained two neural networks for a regression task. 

We can see that both models are able to approximate the underlying sine wave function. 

The model with two hidden layers might be able to capture more complex patterns if the data were more complex, but for this simple case, both models perform well.